## Introduction

This notebook is broken up into sections by object type.

Each section contains four code blocks:

1. Request properties and export to `data/<object_type>_props.json`.
   - This was used to validate property names against the ones shown in the [Cal-ITP: Hubspot CRM Properties](https://docs.google.com/spreadsheets/d/1ZvU4WWS2K1QEtRIk16M83Xnv8X3-wPTB9Mkci_ZLyBE/edit) sheet, or to discover the available properties for the activity objects.
   - Uses the `get_object_props()` function defined in the Start block.
   - It does not need to be run to export actual record data; continue to block 2 for that.
2. Define the properties and associations we want to export.
   - There are some notes in these comments of these blocks regarding properties and associations that have no data, or data we don't necessarily want to export (like full HTML email bodies).
   - Must be run for subsequent blocks to work.
3. Send export request to the [HubSpot Exports API](https://developers.hubspot.com/docs/api-reference/legacy/crm/exports/guide) to get all object data with properties only (no associations).
   - These can be viewed and their data can be downloaded in [the HubSpot dashboard's Import & Export page](https://app.hubspot.com/sales-products-settings/5519226/importexport).
   - This was used to test property names for correctness and see which ones were unused.
   - It does not need to be run if one wants to do a full export (just skip to block 4), but can be run for a simpler test that excludes association data.
4. Send export requests 
   - Uses the `request_exports()` function defined in the Start block.
5. For a group of export requests _just made_ by the above block, monitor their status, download them when complete, and combine them all into a single CSV at `data/<object_type>_data.csv`.
   - Uses the `download_exports_to_csv()` function defined in the Start block.

### Setup

Requires a HubSpot [legacy private app](https://developers.hubspot.com/docs/apps/legacy-apps/private-apps/overview) with every available scope ending in `.read`.

The app will provide an access token that should be stored in an environment variable called `HUBSPOT_ACCESS_TOKEN`.

You can copy the sample environment file to get started; run the following command from the root of this repository:

```bash
cp .env.sample .env
```

Then open `.env` and fill in with your access token.

**The block below should be run before attempting any other blocks in this notebook.**
It will need to be re-run after restarting the kernel, as well.

In [ ]:
import io
import os
import time
import zipfile
from pathlib import Path

from hubspot import HubSpot
from hubspot.crm.exports import PublicExportRequest

import pandas as pd
import requests

from data.utils import hubspot_to_df, write_csv_records, write_json_records


ACCESS_TOKEN = os.environ["HUBSPOT_ACCESS_TOKEN"]
ASSOCIATION_TYPES = ["contacts", "calls", "emails", "meetings", "notes", "tasks", "tickets"]

hubspot = HubSpot(access_token=ACCESS_TOKEN)


class RunMode:
    FULL = 0
    DRY = 1
    LIVE = 2


RUN_MODE = int(os.environ.get("HUBSPOT_API_RUN_MODE", RunMode.DRY))
print(f"Run mode: {RUN_MODE}")

if RUN_MODE == RunMode.FULL:
    # Delete existing data files
    [f.unlink() for f in Path("data").iterdir()]


def get_object_props(object_type: str, nicename: str = ""):
    """Exports details for all properties of an object type to a JSON file."""
    props = hubspot.crm.properties.core_api.get_all(object_type=object_type, archived=False)
    props_df = hubspot_to_df(props)
    if type(props_df) is pd.DataFrame:
        write_json_records(props_df, f"{nicename or object_type}_props.json")


def request_exports(object_type: str, associations: list[list[str]], total_exports: int, core_props: list[str], extra_props: list[str], nicename: str = "") -> list[object]:
    """Requests as many exports as needed for an object type with its associations and returns their confirmation responses.

    Must be done in multiple parts because the HubSpot export API will only accept four associated object types at a time.
    `associations` should be passed as a list of lists of HubSpot object type strings.
    """
    export_requests = []

    for index, association_set in enumerate(associations):
        # Always include the core props, but include the extra props for the first export.
        # This can't just be `props = core_props` because then it will point to the same location in memory
        # and mutate core_props in a way that persists through future iterations.
        props = []
        props += core_props

        if index == 0:
            props += extra_props

        public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name=f"Full {nicename or object_type} export ({index + 1} of {total_exports})", object_type=object_type, object_properties=props, associated_object_type=association_set)

        if RUN_MODE == RunMode.FULL:
            export_requests.append(hubspot.crm.exports.public_exports_api.start(public_export_request))

    return export_requests


def download_exports_to_csv(object_type: str, export_requests: list[object], total_exports: int):
    """Waits for exports to complete, collects exported CSV files, and combines into one CSV file."""
    dfs = []

    for index, export_request in enumerate(export_requests):
        status = None
        timer = 5 * 60  # 5 minutes in seconds
        print(f"Checking status of export request {index + 1} of {total_exports}...", end="")

        while timer > 0:
            time.sleep(5)
            if status := hubspot.crm.exports.public_exports_api.get_status(getattr(export_request, "id")):
                if getattr(status, "status") == "COMPLETE":
                    print("complete! Downloading data.")

                    result = getattr(status, "result")

                    if result and ".zip" in result:
                        # Download the zip file into memory
                        response = requests.get(result)
                        response.raise_for_status() # Ensure the download was successful

                        # Open the zip file from memory
                        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                            # Find the first file inside the zip that ends with '.csv'
                            csv_files = [f for f in z.namelist() if f.endswith('.csv')]

                            if not csv_files:
                                raise ValueError(f"No CSV file found inside the zip archive at {result}")

                            target_csv = csv_files[0]

                            # Open that specific CSV and read it into Pandas
                            with z.open(target_csv) as f:
                                df = pd.read_csv(f)
                    else:
                        # Download export CSV directly into DataFrame
                        storage_options = {'User-Agent': 'Mozilla/5.0'}  # needed to prevent HubSpot from returning a 403
                        df = pd.read_csv(result, storage_options=storage_options)

                    # Dynamically grab the name of the first column
                    first_column = df.columns[0]

                    # Set the first column as the index so Pandas can match rows across files
                    df.set_index(first_column, inplace=True)

                    dfs.append(df)
                    break
            timer -= 5
            print(f".", end="")

    # Combine the DataFrames horizontally (axis=1)
    # This merges them based on the index (the first column we set above)
    combined_df = pd.concat(dfs, axis=1)

    # Drop any duplicated columns
    # .duplicated() flags duplicates as True, the ~ operator inverts it to keep only the unique columns
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]

    # Reset the index to turn the first column back into a regular data column
    combined_df.reset_index(inplace=True)

    # Export the final combined DataFrame to a new CSV file
    output_filename = f"{object_type}_data.csv"
    write_csv_records(combined_df, output_filename)

    print(f"Successfully combined the CSVs and saved to {output_filename}")

## Contacts

In [ ]:
# request properties and export to data/contact_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("contact")

In [ ]:
# define lists of properties and associations that we want to export
contact_core_props = [
    # Automatically included:
    # hs_object_id (always the first column)
    # createdate (second-to-last column before associations, unless explicitly included earlier in prop list)
    # hs_lastmodifieddate (last column before associations, unless explicitly included earlier in prop list)
    "firstname",
    "lastname",
    "email",
]
contact_extra_props = [
    "phone",
    "jobtitle",
    "job_role",
    "notes_last_contacted",
    "district_rep",
    "conferences",
    "hubspot_owner_id",
    "account_manager",
    "address",
    "asset_tag",
    "benefits_monthly_product_subscription",
    "caltrans_districts",
    "city",
    "engagements_last_meeting_booked",
    "hs_marketable_status",
    "hs_object_id",
    "ntd_monthly_ridership_url",
]
contact_associations = [
    [
        "companies",
        "deals",
        "tickets",
        "2-22517187",  # vendors objectTypeId value
    ],
    [
        "contacts",
        "calls",
        "emails",
        "meetings",
    ],
    [
        "notes",
        "tasks",
    ],
]
total_contact_exports = len(contact_associations)  # for reuse later

In [ ]:
# get all contacts with properties only (no associations)
props = contact_core_props + contact_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All contacts with properties", object_type="CONTACTS", object_properties=props)

if RUN_MODE == RunMode.FULL:
    export_request = hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all contact data with associations
contact_export_requests = request_exports("contact", contact_associations, total_contact_exports, contact_core_props, contact_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("contact", contact_export_requests, total_contact_exports)

## Companies

In [ ]:
# request properties and export to data/company_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("company")

In [ ]:
# define lists of properties and associations that we want to export
company_core_props = [
    # Automatically included:
    # hs_object_id (always the first column)
    # createdate (second-to-last column before associations, unless explicitly included earlier in prop list)
    # hs_lastmodifieddate (last column before associations, unless explicitly included earlier in prop list)
    "name",
    "domain",
]
company_extra_props = [
    "address",
    "email",
    "company_type",
    "caltrans_district",
    "airtable_record_id",
    "about_us",
    "account_manager",
    "additional_url",
    "agency_acronym",
    "agency_collect_signage_info_from_riders_additional_detail",
    "agency_collects_info_from_riders_regarding_signage_and_amenities",
    "agency_logo",
    "agency_name",
    "agency_priority",
    "agency_scheduling_challenges",
    "am_support_remix",
    "applicable_fare_discounts",
    "associated_transit_organizations",
    "benefits_form_notes",
    "benefits_marketing_assistance",
    "benefits_onboarding_call",
    "benefits_testing_assistance",
    "board_approval",
    "bus_stop_signs_and_amenities",
    "ca_assembly_district",
    "ca_senate_district",
    "cad_avl_vendor",
    "cal_itp_services",
    "cellular_provider",
    "city",
    "contract_for",
    "contract_url",
    "contracts_received",
    "country",
    "county",
    "createdate",
    "customer_journey_position",
    "customer_testimonial",
    "date_remix_application_received",
    "description_of_caps",
    "desired_launch_timeline",
    "discount_fare_caps",
    "discount_fare_description",
    "discount_fare_structure",
    "discount_group_fare_differences",
    "engagements_last_meeting_booked",
    "engagements_last_meeting_booked_medium",
    "existing_remix_customer",
    "factors_preventing_remix_use",
    "fare_calculation_software",
    "fare_collection_system",
    "fare_validator_vendor",
    "farebox_vendor",
    "first_contact_createdate",
    "first_deal_created_date",
    "flag_for_quarterly_tier_review",
    "fleet_details",
    "fleet_size",
    "google_maps_data",
    "gtfs_vendor",
    "help_with_product_creation",
    "hs_object_id",
    "hubspot_owner_id",
    "itp_id",
    "no_remix_training_reason",
    "notes_last_contacted",
    "notes_last_updated",
    "notes_next_activity_date",
    "ntd_id",
    "ntd_monthly_ridership_url",
    "number_of_stops_on_the_state_highway_network__shn_",
    "organization_type",
    "pathways_to_be_enabled_for_benefits",
    "payment_processor",
    "payment_processor_vendors",
    "permission_to_use_agency_logo_on_websites_",
    "phone",
    "physical_signage_priority_ranking",
    "produce_gtfs_data",
    "product_creation",
    "real_time_digital_signage_priority",
    "recent_deal_close_date",
    "remix_greatest_challenges",
    "remix_hopes",
    "remix_training_yes_no",
    "rider_benches_needs_met",
    "riders_physical_signage_needs_met",
    "riders_real_time_digital_signage_needs_met",
    "riders_shelter_needs_met",
    "route_details",
    "router_onboard",
    "services",
    "shelter_priority_ranking",
    "special_projects",
    "state",
    "tap_agency",
    "telecom_provider",
    "tier",
    "transit_processor",
    "vendor_type",
    "voms",
    "zip",
    "grants_eligibility",
    "apc_vendor",
    "asset_management_vendor",
    "cad_vendor",
    "data_management_and_analytics_vendor",
    "digital_signage_and_information_display_systems_vendor",
    "fleet_management_vendor",
    "funding_sources",
    "funding_sources__other_",
    "gtfs_status",
    "gtfs_use_cases",
    "interest_in_cal_itp_products",
    "next_schedule_change",
    "passenger_information_systems_vendor",
    "predictive_maintenance_vendor",
    "remix_agreement",
    "scheduling_changes",
    "scheduling_vendor",
    "scheduling_with_zeb",
    "technology_assistances",
    "technology_procurement",
    "voms__scheduling_",
    "days_to_close",
]
company_associations = [
    [
        "contacts",
        "deals",
        "tickets",
        "2-22517187",  # vendors objectTypeId value
    ],
    [
        "companies",
        "calls",
        "emails",
        "meetings",
    ],
    [
        "notes",
        "tasks",
    ],
]
total_company_exports = len(company_associations)  # for reuse later

In [ ]:
# get all companies with properties only (no associations)
props = company_core_props + company_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All companies with properties", object_type="COMPANIES", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all company data with associations
company_export_requests = request_exports("company", company_associations, total_company_exports, company_core_props, company_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("company", company_export_requests, total_company_exports)

## Deals

In [ ]:
# request properties and export to data/deal_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("deal")

In [ ]:
# define lists of properties and associations that we want to export
deal_core_props = [
    # Automatically included:
    # hs_object_id (always the first column)
    # createdate (second-to-last column before associations, unless explicitly included earlier in prop list)
    # hs_lastmodifieddate (last column before associations, unless explicitly included earlier in prop list)
    "dealname",
]
deal_extra_props = [
    "pipeline",
    "dealstage",  # not "deal_stage"
    "accepted_card_schemes",
    "hs_lastmodifieddate",
    "preferred_benefits_website_url__sync_",
    "active_benefits_flows",
    "additional_staff",
    "agency_logo__sync_",
    "agency_ntd_id",
    "agency_voms",
    "applicable_discount_types__sync_",
    "benefits_marketing_assistance__sync_",
    "benefits_testing_assistance__sync_",
    "board_approval__sync_",
    "campaign_status",
    "closed_lost_reason",
    "closedate",
    "createdate",
    "date_access_agreement_was_received",
    "dedicated_product_creation_person__sync_",
    "description_of_fare_caps__sync_",
    "desired_launch_timeline__sync_",
    "different_discount_fares__sync_",
    "discount_fare_caps",
    "test_sync",  # "Account Manager"
    "does_agency_have_existing_gtfs_rt_data_",
    "does_the_agency_have_a_discounted_fare_program_",
    "engagements_last_meeting_booked",
    "existing_remix_customer_",
    "fare_type__sync_",
    "funding_source_s__used_for_payments_project",
    "help_with_product_creation__sync_",
    "hs_priority",
    "hs_tag_ids",
    "implementation",
    "in_person_verification__sync_",
    "is_agency_interested_in_exploring_gtfs_rt_contracts_",
    "is_the_agency_fare_free_",
    "latest_status",
    "next_step__digital_signage_pipeline_",
    "notes_last_contacted",
    "notes_last_updated",
    "notes_next_activity_date",
    "other_funding_sources",
    "pathways_to_be_enabled_for_benefits__sync_",
    "reason_benefits_is_n_a",  # "Reason Benefits is N/A"
    "reason_why_participation_was_declined",  # "Reason why participation was declined"
    "remix_agreement",  # "Remix agreement"
    "remix_research_outreach_status___round_one__onboarding_",  # "Remix Research Outreach Status - Round One (onboarding)"
    "requested_pathways_to_enable",  # "Requested Benefits pathways to enable"
    "requests_for_additional_support",  # "Requests for Additional Use Case Support"
    "research_stage",  # "Research Stage"
    "scheduling_software",  # "Scheduling Software"
    "source_of_agency_interest",  # "Source of agency interest"
    "stage_s__of_benefits_onboarding",  # "Benefits Onboarding Stages"
    "survey",  # "Filled out Stop Registry Survey"
    "tap_agency",  # "TAP Agency"
    "targeted_launch_date",  # "Targeted Launch Date"
    "used_msa_contracts",  # "Used MSA contracts"
    "voms__sync_",  # VOMs (sync)"
    "which_digital_signage_vendors_responded_",  # "Which Digital Signage vendors responded?"
    "which_digital_signage_vendors_were_sent_proposal_",  # "Which Digital Signage vendors were sent proposal?"
    "workstream",  # "Workstream"
    "context_for_not_pursuing",  # "Context for not pursuing"
]
deal_associations = [
    [
        "contacts",
        "companies",
        "tickets",
        "2-22517187",  # vendors objectTypeId value
    ],
    [
        "deals",  # There are no deal–deal associations as of 4/14/2026, but leaving in case some are made before final export.
        "calls",
        "emails",
        "meetings",
    ],
    [
        "notes",
        "tasks",
    ],
]
total_deal_exports = len(deal_associations)  # for reuse later

In [ ]:
# get all deals with properties only (no associations)
props = deal_core_props + deal_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All deals with properties", object_type="DEALS", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all deal data with associations
deal_export_requests = request_exports("deal", deal_associations, total_deal_exports, deal_core_props, deal_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("deal", deal_export_requests, total_deal_exports)

## Tickets

In [ ]:
# request properties and export to data/ticket_props.json
# - HAD TO ADD `tickets` SCOPE TO LEGACY APP
if RUN_MODE == RunMode.FULL:
    get_object_props("ticket")

In [ ]:
# define lists of properties and associations that we want to export
ticket_core_props = [
    # Automatically included:
    # hs_object_id (always the first column)
    # createdate (second-to-last column before associations, unless explicitly included earlier in prop list)
    # hs_lastmodifieddate (last column before associations, unless explicitly included earlier in prop list)
    "subject",
]
ticket_extra_props = [
    "hs_pipeline",
    "hs_pipeline_stage",
    "hs_last_closed_date",
    "active_campaign",
    "closed_date",
    "content",
    "createdate",
    "data_creation_stage",
    "how_can_we_help_",
    "how_can_we_help___transit_agency_",
    "hs_file_upload",
    "hs_lastactivitydate",
    "hs_object_id",
    "hs_tag_ids",
    "hubspot_owner_id",
    "is_this_agency_remix_eligible_",
    "is_this_an_mtc_511_agency_",
    "issue_type",
    "last_reply_date",
    "latest_status",
    "n2024_holiday_outreach_campaign_final_status",
    "pads_validators",
    "proactive_reactive",
    "schedule_feed",
    "scheduling_campaign",
    "service_alerts_feed",
    "should_the_am_reach_out_to_encourage_remix_",
    "source_type",
    "support_ticket_subject",
    "team_supporting",
    "ticket_subject_matter",
    "time_to_close",
    "time_to_first_agent_reply",
    "trip_updates_feed",
    "vehicle_positions_feed",
    "what_is_the_caltrans_district_number_",
    "trip_planners_are_notified_by_Caltrans_of_change_in_GTFS_URL",
    "payments_team_is_notified_by_Caltrans_of_change_in_GTFS_URL",
    "other_stakeholders_identified_by_agency_are_notified_by_Caltrans_or_agency_of_change_in_GTFS_URL",
    "notes",
    "ntd_is_notified_by_caltrans_or_agency_re_gtfs_url_updates",  # "ntd_is_notified_by_Caltrans_or_agency_re:_updated_GTFS_URL",
    "mtc_511_is_notified_of_change_in_GTFS_URL",
    "gtfsrt_provider_is_notified_by_caltrans_or_agency_re_updated_gtfs_url",  # "gtfs-rt_provider_is_notified_by_Caltrans_or_agency_re:_updated_GTFS_URL",
    "gtfs_aggregators_are_notified_by_Caltrans_of_change_in_GTFS_URL",
    "deployment_date_for_gtfs_to_be_live",
    "airtable_gtfs_dataset_records_created_for_new_links",
    "airtable_gtfs_service_data_records_set_to_customer_facing",
]
ticket_associations = [
    [
        "contacts",
        "companies",
        "deals",
        "2-22517187",  # vendors objectTypeId value
    ],
    [
        "tickets",
        "calls",
        "emails",
        "meetings",
    ],
    [
        "notes",
        "tasks",
    ],
]
total_ticket_exports = len(ticket_associations)  # for reuse later

In [ ]:
# get all tickets with properties only (no associations)
props = ticket_core_props + ticket_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All tickets with properties", object_type="TICKETS", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all ticket data with associations
ticket_export_requests = request_exports("ticket", ticket_associations, total_ticket_exports, ticket_core_props, ticket_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("ticket", ticket_export_requests, total_ticket_exports)

## Vendors

In [ ]:
# request properties and export to data/vendor_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("2-22517187", nicename="vendor")

In [ ]:
# define lists of properties and associations that we want to export
vendor_core_props = [
    # Automatically included:
    # hs_object_id (always the first column)
    # createdate (second-to-last column before associations, unless explicitly included earlier in prop list)
    # hs_lastmodifieddate (last column before associations, unless explicitly included earlier in prop list)
    "vendor_name",
    "domain",
]
vendor_extra_props = [
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_lastmodifieddate",
    "hs_merged_object_ids",
    "hs_object_id",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_object_source_label",
    "hs_pipeline",
    "hs_pipeline_stage",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_updated_by_user_id",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
    "vendor_details__connectivity",
    "vendor_details__fare_collection",
    "vendor_details__integration",
    "vendor_details__location",
    "vendor_details__onboard_rider_comms",
    "vendor_details__operator_data",
    "vendor_details__regular_bus",
    "vendor_details__safety_tech",
    "vendor_details__transit_operations",
    "vendor_details__zev",
    "vendor_type",
]
vendor_associations = [
    [
        "contacts",
        "companies",
        "deals",
        "tickets",
        # "2-22517187",  # vendors are not associated with other vendors
    ],
    [
        "emails",
        "meetings",
        "notes",
        "tasks",
        #### "calls",  # has association but no data present
    ],
]
total_vendor_exports = len(vendor_associations)  # for reuse later

In [ ]:
# get all vendors with properties only (no associations)
props = vendor_core_props + vendor_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All vendors with properties", object_type="2-22517187", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all vendor data with associations
vendor_export_requests = request_exports("2-22517187", vendor_associations, total_vendor_exports, vendor_core_props, vendor_extra_props, nicename="vendor")

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("vendor", vendor_export_requests, total_vendor_exports)

## Emails

In [ ]:
# request properties and export to data/email_props.json
# - HAD TO ADD `sales-email-read` SCOPE TO LEGACY APP
if RUN_MODE == RunMode.FULL:
    get_object_props("email")

In [ ]:
# define lists of properties and associations that we want to export
email_core_props = [
    "hs_object_id",
]
email_extra_props = [
    # items marked #### do not currently have any data in existing records
    # TODO: Some body previews are truncated. Figure out how to cleanly export full body.
    "hs_all_accessible_team_ids",
    #### "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    "hs_app_id",
    #### "hs_at_mentioned_owner_ids",
    "hs_attachment_ids",
    "hs_body_preview",
    # "hs_body_preview_html",
    "hs_body_preview_is_truncated",
    #### "hs_campaign_guid",
    #### "hs_campaign_guids",
    "hs_created_by",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_direction_and_unique_id",
    "hs_email_associated_contact_id",
    #### "hs_email_attached_video_id",
    #### "hs_email_attached_video_name",
    #### "hs_email_attached_video_opened",
    #### "hs_email_attached_video_watched",
    "hs_email_bcc_email",
    "hs_email_bcc_firstname",
    "hs_email_bcc_lastname",
    "hs_email_bcc_raw",
    "hs_email_bounce_error_detail_message",
    "hs_email_bounce_error_detail_status_code",
    "hs_email_cc_email",
    "hs_email_cc_firstname",
    "hs_email_cc_lastname",
    "hs_email_cc_raw",
    "hs_email_click_count",
    "hs_email_click_rate",
    "hs_email_details",
    "hs_email_direction",
    "hs_email_encoded_email_associations_request",
    "hs_email_error_message",
    "hs_email_facsimile_send_id",
    "hs_email_from_email",
    "hs_email_from_firstname",
    "hs_email_from_lastname",
    "hs_email_from_raw",
    #### "hs_email_has_inline_images_stripped",
    "hs_email_headers",
    # "hs_email_html",
    "hs_email_logged_from",
    "hs_email_media_processing_status",
    "hs_email_member_of_forwarded_subthread",
    "hs_email_message_id",
    #### "hs_email_migrated_via_portal_data_migration",
    #### "hs_email_ms_teams_payload",
    "hs_email_open_count",
    "hs_email_open_rate",
    #### "hs_email_pending_inline_image_ids",
    "hs_email_post_send_status",
    "hs_email_recipient_drop_reasons",
    "hs_email_reply_count",
    "hs_email_reply_rate",
    #### "hs_email_send_event_id",
    #### "hs_email_send_event_id_created",
    "hs_email_sender_email",
    #### "hs_email_sender_firstname",
    #### "hs_email_sender_lastname",
    #### "hs_email_sender_raw",
    "hs_email_sent_count",
    "hs_email_sent_via",
    "hs_email_status",
    #### "hs_email_stripped_attachment_count",
    "hs_email_subject",
    # "hs_email_text",
    "hs_email_thread_id",
    #### "hs_email_thread_summary",
    "hs_email_to_email",
    "hs_email_to_firstname",
    "hs_email_to_lastname",
    "hs_email_to_raw",
    "hs_email_tracker_key",
    #### "hs_email_validation_skipped",
    "hs_engagement_source",
    #### "hs_engagement_source_id",
    "hs_engagements_last_contacted",
    #### "hs_follow_up_action",
    "hs_gdpr_deleted",
    #### "hs_in_reply_to_engagement_id",
    "hs_incoming_email_is_out_of_office",
    "hs_lastmodifieddate",
    #### "hs_merged_object_ids",
    "hs_modified_by",
    "hs_not_tracking_opens_or_clicks",
    "hs_obj_coords",
    "hs_object_source",
    #### "hs_object_source_detail_1",
    #### "hs_object_source_detail_2",
    #### "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_owner_ids_bcc",
    "hs_owner_ids_cc",
    "hs_owner_ids_from",
    "hs_owner_ids_to",
    "hs_owning_teams",
    #### "hs_pinned_engagement_id",
    #### "hs_product_name",
    #### "hs_queue_membership_ids",
    #### "hs_read_only",
    "hs_scs_association_status",
    "hs_scs_audit_id",
    #### "hs_sequence_id",
    #### "hs_shared_team_ids",
    #### "hs_shared_user_ids",
    "hs_template_id",
    "hs_ticket_create_date",
    "hs_timestamp",
    #### "hs_unique_creation_key",
    "hs_unique_id",
    "hs_unique_tracker_key",
    "hs_updated_by_user_id",
    #### "hs_user_ids_of_all_notification_followers",
    #### "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    #### "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
email_associations = [
    [
        "contacts",
        "companies",
        "deals",
        "2-22517187",  # vendors objectTypeId value
    ],
    [
        "tickets",
        "meetings",
        # "calls",  # no assocation with emails
        # "emails",  # no association with emails
        # "note÷s",  # no association with emails
        # "tasks",  # no association with emails
    ],
]
total_email_exports = len(email_associations)  # for reuse later

In [ ]:
# get all emails with properties only (no associations)
props = email_core_props + email_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All emails with properties (but not full bodies)", object_type="EMAILS", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all email data with associations
email_export_requests = request_exports("email", email_associations, total_email_exports, email_core_props, email_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("email", email_export_requests, total_email_exports)

## Calls

In [ ]:
# request properties and export to data/call_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("call")

In [ ]:
# define lists of properties and associations that we want to export
call_core_props = [
    "hs_object_id",
]
call_extra_props = [
    # items marked #### do not currently have any data in existing records
    "hs_activity_type",
    "hs_all_accessible_team_ids",
    #### "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    #### "hs_at_mentioned_owner_ids",
    #### "hs_attachment_ids",
    "hs_body_preview",
    # "hs_body_preview_html",
    "hs_body_preview_is_truncated", # none are truncated
    #### "hs_call_app_id",
    #### "hs_call_authed_url_provider",
    # "hs_call_body",  # content with newlines
    "hs_call_callee_object_id",
    "hs_call_callee_object_type",
    "hs_call_deal_stage_during_call",
    "hs_call_direction",
    "hs_call_disposition",
    "hs_call_duration",
    "hs_call_external_account_id",
    "hs_call_external_id",
    "hs_call_from_number",
    #### "hs_call_from_number_nickname",
    #### "hs_call_has_transcript",
    "hs_call_has_voicemail",
    "hs_call_location",
    #### "hs_call_meeting_event_id",
    #### "hs_call_meeting_id",
    #### "hs_call_ms_teams_payload",
    "hs_call_phone_number_source",
    "hs_call_primary_deal",
    #### "hs_call_recording_duration",
    #### "hs_call_recording_url",
    "hs_call_routing_strategy_type",
    "hs_call_source",
    "hs_call_status",
    #### "hs_call_suggested_next_actions",
    #### "hs_call_summary",
    #### "hs_call_summary_execution_id",
    "hs_call_title",
    "hs_call_to_number",
    #### "hs_call_to_number_nickname",
    #### "hs_call_transcript_audio_num_channels",
    #### "hs_call_transcript_tracked_terms",
    #### "hs_call_transcription_id",
    "hs_call_unified_phone_number_id",
    #### "hs_call_unique_external_id",
    #### "hs_call_video_meeting_type",
    #### "hs_call_video_recording_url",
    #### "hs_call_zoom_meeting_uuid",
    "hs_calls_service_call_id",
    #### "hs_campaign_guid",
    #### "hs_campaign_guids",
    "hs_connected_count",
    "hs_created_by",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_engagement_source",
    "hs_engagement_source_id",
    "hs_engagements_last_contacted",
    #### "hs_follow_up_action",
    #### "hs_gdpr_deleted",
    #### "hs_hosted_video_conference_url",
    #### "hs_is_voice_agent_call",
    "hs_lastmodifieddate",
    #### "hs_merged_object_ids",
    "hs_modified_by",
    "hs_obj_coords",
    "hs_object_source",
    #### "hs_object_source_detail_1",
    #### "hs_object_source_detail_2",
    #### "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_owning_teams",
    #### "hs_pinned_engagement_id",
    #### "hs_product_name",
    #### "hs_queue_membership_ids",
    #### "hs_read_only",
    #### "hs_shared_team_ids",
    #### "hs_shared_user_ids",
    "hs_timestamp",
    #### "hs_unique_creation_key",
    "hs_unique_id",
    #### "hs_unknown_visitor_conversation",
    "hs_updated_by_user_id",
    #### "hs_user_ids_of_all_notification_followers",
    #### "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    #### "hs_voice_agent_caller_requested_live_agent",
    #### "hs_voice_agent_escalated_to_live_agent",
    #### "hs_voice_agent_is_most_recent_customer_feedback_positive",
    "hs_voicemail_count",
    #### "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
call_associations = [
    [
        "contacts",
        "companies",
        "deals",
        "tickets",
        #### "2-22517187",  # vendors objectTypeId value -- has association but no data present
    ],
    [
        "meetings",
        #### "calls",  # has association but no data present
        # "tasks",  # no association with calls
        # "notes",  # no association with calls
        # "emails",  # no association with calls
    ],
]
total_call_exports = len(call_associations)  # for reuse later

In [ ]:
# Property check
props = call_core_props + call_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All calls with properties (but not HTML bodies)", object_type="CALLS", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all call data with associations
call_export_requests = request_exports("call", call_associations, total_call_exports, call_core_props, call_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("call", call_export_requests, total_call_exports)

## Meetings

In [ ]:
# request properties and export to data/meeting_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("meeting")

In [ ]:
# define lists of properties and associations that we want to export
meeting_core_props = [
    "hs_object_id",
]
meeting_extra_props = [
    # items marked #### do not currently have any data in existing records
    "hs_activity_type",
    "hs_all_accessible_team_ids",
    "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    #### "hs_associated_call_source",
    "hs_at_mentioned_owner_ids",
    "hs_attachment_ids",
    "hs_attendee_owner_ids",
    "hs_body_preview",
    # "hs_body_preview_html",
    "hs_body_preview_is_truncated",
    #### "hs_booking_instance_id",
    #### "hs_campaign_guid",
    #### "hs_campaign_guids",
    "hs_contact_first_outreach_date",
    "hs_created_by",
    "hs_created_by_scheduling_page",
    "hs_created_by_user_id",
    "hs_createdate",
    #### "hs_customized_reminder_body_format",
    #### "hs_customized_reminder_subject_format",
    "hs_engagement_source",
    "hs_engagement_source_id",
    "hs_engagements_last_contacted",
    #### "hs_external_calendar_id",
    #### "hs_follow_up_action",
    #### "hs_follow_up_context",
    "hs_follow_up_tasks_remaining",
    "hs_gdpr_deleted",
    #### "hs_guest_emails",
    #### "hs_has_associated_notetaker_call_transcript",
    "hs_has_call_with_transcript",
    "hs_i_cal_uid",
    "hs_include_description_in_reminder",
    "hs_internal_meeting_notes",  # HTML, but there's no non-HTML version
    "hs_lastmodifieddate",
    # "hs_meeting_body",  # some of these have newlines, some don't
    "hs_meeting_calendar_event_hash",
    "hs_meeting_change_id",
    "hs_meeting_created_from_link_id",
    "hs_meeting_end_time",
    "hs_meeting_external_url",
    "hs_meeting_location",
    "hs_meeting_location_type",
    #### "hs_meeting_ms_teams_payload",
    #### "hs_meeting_notetaker_id",
    #### "hs_meeting_notetaker_selection",
    "hs_meeting_outcome",
    #### "hs_meeting_payments_session_id",
    #### "hs_meeting_pre_meeting_prospect_reminders",
    "hs_meeting_source",
    "hs_meeting_source_id",
    "hs_meeting_start_time",
    "hs_meeting_title",
    "hs_meeting_web_conference_meeting_id",
    #### "hs_merged_object_ids",
    "hs_modified_by",
    #### "hs_notetaker_attendance_duration",
    "hs_obj_coords",
    "hs_object_source",
    #### "hs_object_source_detail_1",
    #### "hs_object_source_detail_2",
    #### "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_outcome_canceled_count",
    "hs_outcome_completed_count",
    "hs_outcome_no_show_count",
    "hs_outcome_rescheduled_count",
    "hs_outcome_scheduled_count",
    "hs_owning_teams",
    #### "hs_pinned_engagement_id",
    #### "hs_product_name",
    #### "hs_queue_membership_ids",
    #### "hs_read_only",
    "hs_recurring_event_schedule",
    "hs_recurring_event_series_id",
    "hs_recurring_event_type",
    "hs_recurring_event_unique_lookup",
    #### "hs_roster_object_coordinates",
    "hs_scheduled_tasks",
    #### "hs_shared_team_ids",
    #### "hs_shared_user_ids",
    "hs_time_to_book_meeting_from_first_contact",
    "hs_timestamp",
    "hs_timezone",
    #### "hs_unique_creation_key",
    "hs_unique_id",
    "hs_updated_by_user_id",
    #### "hs_user_ids_of_all_notification_followers",
    #### "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    "hs_video_conference_platform",
    "hs_video_conference_url",
    "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
meeting_associations = [
    [
        "contacts",
        "companies",
        "deals",
        "tickets",
    ],
    [
        "2-22517187",  # vendors objectTypeId value
        "calls",
        "tasks",  # has association but no data present
        "emails",  # has association but no data present
        # "meetings",  # no association with other meetings
        # "notes",  # no association with meetings
    ],
]
total_meeting_exports = len(meeting_associations)  # for reuse later

In [ ]:
# get all emails with properties only (no associations)
props = meeting_core_props + meeting_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All meetings with properties (but not HTML bodies)", object_type="MEETINGS", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all meeting data with associations
meeting_export_requests = request_exports("meeting", meeting_associations, total_meeting_exports, meeting_core_props, meeting_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("meeting", meeting_export_requests, total_meeting_exports)

## Notes

In [ ]:
# request properties and export to data/note_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("note")

In [ ]:
# define lists of properties and associations that we want to export
note_core_props = [
    "hs_object_id",
]
note_extra_props = [
    # items marked #### do not currently have any data in existing records
    "hs_all_accessible_team_ids",
    #### "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    "hs_at_mentioned_owner_ids",
    #### "hs_attachment_from_form_submission",
    "hs_attachment_ids",
    "hs_body_preview",
    # "hs_body_preview_html",
    "hs_body_preview_is_truncated",
    "hs_created_by",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_engagement_source",
    "hs_engagement_source_id",
    #### "hs_follow_up_action",
    "hs_gdpr_deleted",
    "hs_hd_ticket_ids",
    "hs_lastmodifieddate",
    #### "hs_merged_object_ids",
    "hs_modified_by",
    "hs_note_body",
    #### "hs_note_ms_teams_payload",
    "hs_obj_coords",
    "hs_object_source",
    "hs_object_source_detail_1",
    #### "hs_object_source_detail_2",
    #### "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_owning_teams",
    #### "hs_pinned_engagement_id",
    #### "hs_product_name",
    #### "hs_queue_membership_ids",
    #### "hs_read_only",
    #### "hs_shared_team_ids",
    #### "hs_shared_user_ids",
    "hs_timestamp",
    #### "hs_unique_creation_key",
    #### "hs_unique_id",
    "hs_updated_by_user_id",
    #### "hs_user_ids_of_all_notification_followers",
    #### "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
note_associations = [
    [
        "contacts",
        "companies",
        "deals",
        "tickets",
    ],
    [
        "2-22517187",  # vendors objectTypeId value
    ],
    # [
    #     # "notes",  # no association with other notes
    #     # "emails",  # no association with notes
    #     # "calls",  # no association with notes
    #     # "meetings",  # no association with notes
    #     # "tasks",  # no association with notes
    # ],
]
total_note_exports = len(note_associations)  # for reuse later

In [ ]:
# get all notes with properties only (no associations)
props = note_core_props + note_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All notes with properties (but not HTML bodies)", object_type="NOTES", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all note data with associations
note_export_requests = request_exports("note", note_associations, total_note_exports, note_core_props, note_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("note", note_export_requests, total_note_exports)

## Tasks

In [ ]:
# request properties and export to data/task_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("task")

In [ ]:
# define lists of properties and associations that we want to export
task_core_props = [
    "hs_object_id",
]
task_extra_props = [
    # items marked #### do not currently have any data in existing records
    "hs_all_accessible_team_ids",
    #### "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    "hs_associated_company_labels",
    "hs_associated_contact_labels",
    "hs_at_mentioned_owner_ids",
    #### "hs_attachment_ids",
    "hs_body_preview",
    # "hs_body_preview_html",
    "hs_body_preview_is_truncated",
    #### "hs_calendar_event_id",
    "hs_created_by",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_date_entered_60b5c368_04c4_4d32_9b4a_457e159f49b7_13292096",
    "hs_date_entered_61bafb31_e7fa_46ed_aaa9_1322438d6e67_1866552342",
    "hs_date_entered_af0e6a5c_2ea3_4c72_b69f_7c6cb3fdb591_1652950531",
    "hs_date_entered_dd5826e4_c976_4654_a527_b59ada542e52_2144133616",
    "hs_date_entered_fc8148fb_3a2d_4b59_834e_69b7859347cb_1813133675",
    "hs_date_exited_60b5c368_04c4_4d32_9b4a_457e159f49b7_13292096",
    "hs_date_exited_61bafb31_e7fa_46ed_aaa9_1322438d6e67_1866552342",
    "hs_date_exited_af0e6a5c_2ea3_4c72_b69f_7c6cb3fdb591_1652950531",
    "hs_date_exited_dd5826e4_c976_4654_a527_b59ada542e52_2144133616",
    "hs_date_exited_fc8148fb_3a2d_4b59_834e_69b7859347cb_1813133675",
    "hs_engagement_source",
    "hs_engagement_source_id",
    "hs_engagements_last_contacted",
    #### "hs_follow_up_action",
    "hs_gdpr_deleted",
    "hs_lastmodifieddate",
    #### "hs_marketing_studio_node_id",
    #### "hs_marketing_task_category",
    #### "hs_marketing_task_category_id",
    #### "hs_merged_object_ids",
    "hs_modified_by",
    #### "hs_msteams_message_id",
    "hs_obj_coords",
    "hs_object_id",
    "hs_object_source",
    "hs_object_source_detail_1",
    #### "hs_object_source_detail_2",
    #### "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_owning_teams",
    #### "hs_partner_map_category",
    #### "hs_partner_map_year",
    #### "hs_pinned_engagement_id",
    "hs_pipeline",
    "hs_pipeline_stage",
    #### "hs_product_name",
    "hs_queue_membership_ids",
    #### "hs_read_only",
    "hs_repeat_status",
    "hs_scheduled_tasks",
    #### "hs_shared_team_ids",
    #### "hs_shared_user_ids",
    "hs_start_date",
    "hs_start_date_with_fallback",
    "hs_target_duration",
    "hs_task_assigned_by_user_id",
    #### "hs_task_assigned_contacts",
    #### "hs_task_blocked_task_ids",
    #### "hs_task_blocking_task_ids",
    "hs_task_body",    # content with newlines
    #### "hs_task_campaign_guid",
    "hs_task_company_domain",
    "hs_task_company_industry",
    #### "hs_task_company_is_target_account",
    "hs_task_completion_count",
    "hs_task_completion_date",
    "hs_task_contact_email",
    "hs_task_contact_job_title",
    "hs_task_contact_phone",
    "hs_task_contact_timezone",
    "hs_task_family",
    "hs_task_for_object_type",
    "hs_task_is_all_day",
    "hs_task_is_completed",
    "hs_task_is_completed_call",
    "hs_task_is_completed_email",
    "hs_task_is_completed_linked_in",
    "hs_task_is_completed_sequence",
    "hs_task_is_open",
    "hs_task_is_overdue",
    "hs_task_is_past_due_date",
    "hs_task_is_sub_task",
    "hs_task_last_contact_outreach",
    "hs_task_last_sales_activity_timestamp",
    "hs_task_missed_due_date",
    "hs_task_missed_due_date_count",
    #### "hs_task_ms_teams_payload",
    #### "hs_task_parent_task_associated_objects",
    #### "hs_task_parent_task_id",
    #### "hs_task_parent_task_owner_ids",
    "hs_task_priority",
    "hs_task_probability_to_complete",
    "hs_task_relative_reminders",
    "hs_task_reminders",
    "hs_task_repeat_interval",
    "hs_task_send_default_reminder",
    #### "hs_task_sequence_enrollment_active",
    #### "hs_task_sequence_id",
    #### "hs_task_sequence_step_enrollment_contact_id",
    #### "hs_task_sequence_step_enrollment_id",
    #### "hs_task_sequence_step_number",
    #### "hs_task_sequence_step_order",
    "hs_task_status",
    #### "hs_task_sub_task_at_mentioned_owner_ids",
    #### "hs_task_sub_task_ids",
    #### "hs_task_sub_task_owner_ids",
    "hs_task_subject",
    "hs_task_template_id",
    "hs_task_type",
    #### "hs_task_uncompleted_sub_task_ids",
    "hs_time_in_60b5c368_04c4_4d32_9b4a_457e159f49b7_13292096",
    "hs_time_in_61bafb31_e7fa_46ed_aaa9_1322438d6e67_1866552342",
    "hs_time_in_af0e6a5c_2ea3_4c72_b69f_7c6cb3fdb591_1652950531",
    "hs_time_in_dd5826e4_c976_4654_a527_b59ada542e52_2144133616",
    "hs_time_in_fc8148fb_3a2d_4b59_834e_69b7859347cb_1813133675",
    "hs_timestamp",
    #### "hs_unique_creation_key",
    "hs_unique_id",
    "hs_universal_association_selector",
    "hs_updated_by_user_id",
    #### "hs_user_ids_of_all_notification_followers",
    #### "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
task_associations = [
    [
        "contacts",
        "companies",
        "deals",
        "tickets",
    ],
    [
        "2-22517187",  # vendors objectTypeId value
        "meetings",  # has association but no data present
        "tasks",  # has association but no data present
    ],
    # [
    #     # "calls",  # no association with tasks
    #     # "emails",  # no association with tasks
    #     # "notes",  # no association with tasks
    # ],
]
total_task_exports = len(task_associations)  # for reuse later

In [ ]:
# get all tasks with properties only (no associations)
props = task_core_props + task_extra_props

public_export_request = PublicExportRequest(export_type="VIEW", format="CSV", export_name="All tasks with properties", object_type="TASKS", object_properties=props)

if RUN_MODE == RunMode.FULL:
    hubspot.crm.exports.public_exports_api.start(public_export_request)

In [ ]:
# request exports for all task data with associations
task_export_requests = request_exports("task", task_associations, total_task_exports, task_core_props, task_extra_props)

In [ ]:
# wait for exports to complete, collect exported CSV files, and combine
download_exports_to_csv("task", task_export_requests, total_task_exports)